<a href="https://colab.research.google.com/github/KaviduR320/Statistical-Learning-e23306/blob/main/Assignment%2304.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [4]:
import io
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy.stats as stats
from google.colab import files
from plotly.subplots import make_subplots
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder, OrdinalEncoder,
                                   RobustScaler, StandardScaler)

warnings.filterwarnings('ignore')

# -------------------------------------------------------------
# GLOBAL VISUAL CONFIGURATION
# -------------------------------------------------------------
class VisualConfig:
    THEME = "plotly_dark"
    COLOR_PRIMARY = "#118ab2"     # Soft Teal
    COLOR_SECONDARY = "#ef476f"   # Muted Rose
    COLOR_ACCENT = "#ffd166"      # Soft Gold
    COLOR_NEUTRAL = "#073b4c"     # Dark Navy
    DIVERGING_SCALE = "Electric"  # Premium look for heatmaps
    SEQUENTIAL_SCALE = "Tealrose"

# -------------------------------------------------------------
# REFACTORED PLOTTING PIPELINE
# -------------------------------------------------------------
class InteractiveRenderer:
    """Handles static and standalone HTML components for reporting."""

    @classmethod
    def render_bar(cls, df, x_axis, y_axis=None, title="Distribution"):
        fig = px.bar(
            df, x=x_axis, y=y_axis, title=title,
            template=VisualConfig.THEME,
            color_discrete_sequence=[VisualConfig.COLOR_PRIMARY]
        )
        return fig.to_html(full_html=False)

    @classmethod
    def render_pie(cls, df, category, metric, title="Proportional Split"):
        fig = px.pie(
            df, names=category, values=metric, title=title,
            template=VisualConfig.THEME,
            color_discrete_sequence=[VisualConfig.COLOR_PRIMARY, VisualConfig.COLOR_SECONDARY, VisualConfig.COLOR_ACCENT]
        )
        return fig.to_html(full_html=False)

    @classmethod
    def render_histogram(cls, df, feature, bins=35, title="Density Distribution"):
        fig = px.histogram(
            df, x=feature, nbins=bins, title=title,
            template=VisualConfig.THEME,
            color_discrete_sequence=[VisualConfig.COLOR_SECONDARY]
        )
        return fig.to_html(full_html=False)


# -------------------------------------------------------------
# CORE INTELLIGENT DATA PROCESSING ENGINE
# -------------------------------------------------------------
class AdvancedDataArchitect:
    """Advanced Data Inspection, Engineering, and Analytical Visualizer Engine."""

    def __init__(self):
        self.dataframe = None
        self.source_dataframe = None
        self.numeric_features = []
        self.categorical_features = []
        self.execution_logs = []

    def load_user_file(self):
        """Asks user for file upload via Google Colab environments."""
        print("💡 Waiting for CSV Upload...")
        uploaded_files = files.upload()

        for filename in uploaded_files.keys():
            self.dataframe = pd.read_csv(io.BytesIO(uploaded_files[filename]))
            self.source_dataframe = self.dataframe.copy()
            self._log(f"Successfully mounted {filename}. Matrix dimensions: {self.dataframe.shape}")
            self._refresh_feature_types()
            return self.dataframe

    def _log(self, message):
        self.execution_logs.append(message)
        print(f"[PROCESS LOG] {message}")

    def _refresh_feature_types(self):
        """Discovers numeric vs nominal variables dynamically."""
        self.numeric_features = self.dataframe.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_features = self.dataframe.select_dtypes(include=['object', 'category']).columns.tolist()

    def clean_corrupted_tokens(self):
        """Replaces common unstructured string patterns into structured NaNs."""
        garbage_tokens = ['?', 'n/a', 'NULL', '', 'None', 'null', 'NaN', 'nan', 'unknown', 'UNKNOWN']
        self.dataframe.replace(garbage_tokens, np.nan, inplace=True)

        # Smart type inference optimization
        for col in self.dataframe.columns:
            if self.dataframe[col].dtype == 'object':
                try:
                    inferred_numeric = pd.to_numeric(self.dataframe[col])
                    if not inferred_numeric.isna().all():
                        self.dataframe[col] = inferred_numeric
                        self._log(f"Inferred numeric schema for column: '{col}'")
                except (ValueError, TypeError):
                    pass
        self._refresh_feature_types()

    def generate_structural_audit(self):
        """Prints a highly comprehensive structural overview of the active dataset."""
        print("\n" + "⚡" * 30)
        print("                  DATASET STRUCTURAL REPORT                 ")
        print("⚡" * 30)
        print(f"• Dynamic Shapes  : {self.dataframe.shape[0]:,} records | {self.dataframe.shape[1]:,} variables")
        print(f"• Numerical Pool  : {len(self.numeric_features)} features")
        print(f"• Categorical Pool: {len(self.categorical_features)} features")

        print("\n--- SAMPLE FRAMEWORK (TOP 5 DEEP INSPECTION) ---")
        print(self.dataframe.head(5))

        if self.numeric_features:
            print("\n--- MATHEMATICAL MOMENTS SUMMARY ---")
            print(self.dataframe[self.numeric_features].describe().T)

        print("\n--- CATEGORICAL DENSITY CHECKS ---")
        for col in self.categorical_features:
            print(f"  ↳ Feature [{col}]: {self.dataframe[col].nunique()} discrete categories")

    def smart_imputation_engine(self, strategy='auto', manual_fill=None):
        """
        Advanced imputation engine.
        'auto' strategy applies structural modes to categories and medians to skewed variables.
        """
        self.clean_corrupted_tokens()
        self._log(f"Executing '{strategy}' missing value resolutions...")

        for col in self.dataframe.columns:
            null_count = self.dataframe[col].isna().sum()
            if null_count > 0:
                if col in self.numeric_features:
                    if strategy == 'auto' or strategy == 'median':
                        fill_value = self.dataframe[col].median()
                    elif strategy == 'mean':
                        fill_value = self.dataframe[col].mean()
                    elif strategy == 'constant' and manual_fill is not None:
                        fill_value = manual_fill
                    else:
                        fill_value = self.dataframe[col].mode()[0] if not self.dataframe[col].mode().empty else 0

                    self.dataframe[col].fillna(fill_value, inplace=True)
                else:
                    # Categorical fallbacks
                    fill_value = self.dataframe[col].mode()[0] if not self.dataframe[col].mode().empty else 'Unspecified'
                    if strategy == 'constant' and manual_fill is not None:
                        fill_value = manual_fill
                    self.dataframe[col].fillna(fill_value, inplace=True)

        self._log(f"Imputation completed. Total systemic NaNs remaining: {self.dataframe.isna().sum().sum()}")
        return self.dataframe

    def purge_duplicate_records(self):
        pre_count = len(self.dataframe)
        self.dataframe.drop_duplicates(inplace=True)
        self._log(f"Purged {pre_count - len(self.dataframe)} duplicate records from memory.")
        return self.dataframe

    def manage_outlier_variance(self, target_cols=None, action='cap', bounds_multiplier=1.5):
        """
        Advanced Outlier handler.
        Supports structural 'flagging', total 'delete', or 'cap' (Winsorization/soft capping).
        """
        if target_cols is None:
            target_cols = self.numeric_features

        master_outlier_mask = pd.Series([False] * len(self.dataframe), index=self.dataframe.index)

        for col in target_cols:
            if col in self.numeric_features:
                q25, q75 = self.dataframe[col].quantile(0.25), self.dataframe[col].quantile(0.75)
                iqr = q75 - q25
                lower_threshold = q25 - (bounds_multiplier * iqr)
                upper_threshold = q75 + (bounds_multiplier * iqr)

                feature_mask = (self.dataframe[col] < lower_threshold) | (self.dataframe[col] > upper_threshold)
                master_outlier_mask |= feature_mask

                if action == 'cap':
                    # Advanced capping logic (Winsorization)
                    self.dataframe[col] = np.clip(self.dataframe[col], lower_threshold, upper_threshold)

        if action == 'flag':
            self.dataframe['anomaly_flag'] = master_outlier_mask.astype(int)
            self._log(f"Flagged {master_outlier_mask.sum()} records as anomalous variance.")
        elif action == 'delete':
            self.dataframe = self.dataframe[~master_outlier_mask]
            self._log(f"Dropped {master_outlier_mask.sum()} outlier entries from structural framework.")
        elif action == 'cap':
            self._log(f"Applied mathematical capping boundaries across selected numeric inputs.")

        return self.dataframe

    def drop_features(self, string_inputs):
        targets = [token.strip() for token in string_inputs.split(',') if token.strip() in self.dataframe.columns]
        self.dataframe.drop(columns=targets, inplace=True, errors='ignore')
        self._log(f"Dropped targeted columns: {targets}")
        self._refresh_feature_types()
        return self.dataframe

    def scale_numerical_space(self, technique='standard', feature_subsets=None):
        targets = feature_subsets or self.numeric_features
        if not targets:
            return pd.DataFrame()

        engines = {
            'minmax': MinMaxScaler(),
            'standard': StandardScaler(),
            'robust': RobustScaler()
        }

        selected_engine = engines.get(technique, StandardScaler())
        transformed_arrays = selected_engine.fit_transform(self.dataframe[targets])

        output_df = pd.DataFrame(
            transformed_arrays,
            columns=[f"scaled_{c}" for c in targets],
            index=self.dataframe.index
        )
        self._log(f"Extracted {technique}-scaled vector space for {len(targets)} variables.")
        return output_df

    def encode_categorical_space(self, technique='onehot', feature_subsets=None):
        targets = feature_subsets or self.categorical_features
        if not targets:
            return pd.DataFrame()

        if technique == 'onehot':
            encoded_df = pd.get_dummies(self.dataframe[targets], prefix=[f"enc_{c}" for c in targets], drop_first=True)
            self._log(f"One-Hot transformation yielded vector matrix dimensions: {encoded_df.shape}")
            return encoded_df

        elif technique == 'ordinal':
            ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            transformed = ord_encoder.fit_transform(self.dataframe[targets].astype(str))
            encoded_df = pd.DataFrame(transformed, columns=[f"ord_{c}" for c in targets], index=self.dataframe.index)
            self._log(f"Ordinal encoding applied to nominal subsets.")
            return encoded_df
        else:
            raise ValueError("Supported encoding metrics include: 'onehot', 'ordinal'")

    def merge_transformed_matrices(self, frame_a=None, frame_b=None):
        consolidated = self.dataframe.copy()
        if frame_a is not None and not frame_a.empty:
            consolidated = pd.concat([consolidated, frame_a], axis=1)
        if frame_b is not None and not frame_b.empty:
            consolidated = pd.concat([consolidated, frame_b], axis=1)
        self._log(f"Integration finalized. Combined analytical frame matrix: {consolidated.shape}")
        return consolidated

    # -------------------------------------------------------------
    # ADVANCED GRAPHICAL ENGINES (THEMED)
    # -------------------------------------------------------------
    def visualization_univariate_grid(self, target_features=None):
        """Renders stylized subplots with modern violin distributions and histograms."""
        targets = target_features or self.numeric_features[:2]

        for feature in targets:
            fig = make_subplots(
                rows=1, cols=3,
                subplot_titles=(f"Density Structure", f"Index Dispersion Map", f"Frequency Limits")
            )

            # Violin plots
            fig.add_trace(
                go.Violin(y=self.dataframe[feature], name=feature, box_visible=True,
                          meanline_visible=True, fillcolor=VisualConfig.COLOR_PRIMARY, line_color="#ffffff"),
                row=1, col=1
            )
            # Scatter/Index plots
            fig.add_trace(
                go.Scatter(x=self.dataframe.index, y=self.dataframe[feature], mode='markers',
                           marker=dict(color=self.dataframe[feature], colorscale=VisualConfig.SEQUENTIAL_SCALE, size=5, opacity=0.7)),
                row=1, col=2
            )
            # Histograms
            fig.add_trace(
                go.Histogram(x=self.dataframe[feature], nbinsx=25, marker_color=VisualConfig.COLOR_SECONDARY),
                row=1, col=3
            )

            fig.update_layout(
                height=450,
                title_text=f"📊 Advanced Diagnostics Framework: {feature}",
                template=VisualConfig.THEME,
                showlegend=False
            )
            fig.show()

    def visualization_bivariate_relationship(self, var_x, var_y):
        """Plots bivariate relationships using clean styling options based on input types."""
        is_x_num = var_x in self.numeric_features
        is_y_num = var_y in self.numeric_features

        if is_x_num and is_y_num:
            r_coef = self.dataframe[var_x].corr(self.dataframe[var_y])
            fig = px.scatter(
                self.dataframe, x=var_x, y=var_y, trendline='ols',
                title=f"Scatter Metric: {var_x} vs {var_y} (Pearson R: {r_coef:.3f})",
                template=VisualConfig.THEME,
                color_discrete_sequence=[VisualConfig.COLOR_PRIMARY]
            )
            fig.update_traces(marker=dict(size=7, opacity=0.8, line=dict(width=0.5, color='white')))
            fig.show()

        elif (is_x_num and not is_y_num) or (not is_x_num and is_y_num):
            numeric_target = var_x if is_x_num else var_y
            categorical_target = var_y if is_x_num else var_x

            fig = px.box(
                self.dataframe, x=categorical_target, y=numeric_target,
                title=f"Variance Spread: {numeric_target} split by {categorical_target}",
                template=VisualConfig.THEME,
                color=categorical_target,
                color_discrete_sequence=px.colors.qualitative.Pastel
            )
            fig.show()
        else:
            cross_tabulation = pd.crosstab(self.dataframe[var_x], self.dataframe[var_y])
            fig = px.bar(
                cross_tabulation, barmode='group',
                title=f"Contingency Crosstab Profile: {var_x} versus {var_y}",
                template=VisualConfig.THEME,
                color_discrete_sequence=px.colors.qualitative.Bold
            )
            fig.show()

    def visualization_categorical_distribution(self, feature, top_cutoff=10):
        """Displays formatted distributions for high/low cardinality targets."""
        counts = self.dataframe[feature].value_counts().head(top_cutoff)
        proportions = (counts / len(self.dataframe) * 100).round(2)

        fig = go.Figure(data=[
            go.Bar(
                x=counts.index.astype(str), y=counts.values,
                text=[f"N={v}<br>{p}%" for v, p in zip(counts.values, proportions)],
                textposition='outside',
                marker=dict(color=counts.values, colorscale=VisualConfig.SEQUENTIAL_SCALE)
            )
        ])
        fig.update_layout(
            title=f"Categorical Density Top Counts: {feature}",
            xaxis_title=feature, yaxis_title="Calculated Absolute Frequency",
            template=VisualConfig.THEME
        )
        fig.show()

    def visualization_association_matrix(self):
        """Computes a matrix combining Pearson correlations and Cramer's V metrics."""
        features_pool = self.numeric_features + self.categorical_features
        matrix_dimension = len(features_pool)
        similarity_grid = np.zeros((matrix_dimension, matrix_dimension))

        for r_idx, feat_a in enumerate(features_pool):
            for c_idx, feat_b in enumerate(features_pool):
                if r_idx == c_idx:
                    similarity_grid[r_idx, c_idx] = 1.0
                elif r_idx > c_idx:
                    similarity_grid[r_idx, c_idx] = similarity_grid[c_idx, r_idx]
                else:
                    # Numeric to Numeric
                    if feat_a in self.numeric_features and feat_b in self.numeric_features:
                        similarity_grid[r_idx, c_idx] = self.dataframe[feat_a].corr(self.dataframe[feat_b])

                    # Categorical to Categorical
                    elif feat_a in self.categorical_features and feat_b in self.categorical_features:
                        contingency = pd.crosstab(self.dataframe[feat_a], self.dataframe[feat_b])
                        chi2_val = stats.chi2_contingency(contingency)[0]
                        total_elements = contingency.sum().sum()
                        lower_dimension = min(contingency.shape) - 1
                        similarity_grid[r_idx, c_idx] = np.sqrt(chi2_val / (total_elements * lower_dimension)) if lower_dimension > 0 else 0

                    # Hybrid Association Mapping
                    else:
                        num_f = feat_a if feat_a in self.numeric_features else feat_b
                        cat_f = feat_b if feat_a in self.numeric_features else feat_a
                        dummies = pd.get_dummies(self.dataframe[cat_f], drop_first=True)
                        if not dummies.empty:
                            coef = self.dataframe[num_f].corr(dummies.iloc[:, 0])
                        else:
                            coef = 0
                        similarity_grid[r_idx, c_idx] = abs(coef) if not np.isnan(coef) else 0

        fig = px.imshow(
            similarity_grid, x=features_pool, y=features_pool,
            color_continuous_scale=VisualConfig.DIVERGING_SCALE,
            text_auto='.2f', title="📊 Systemic Correlation & Association Map"
        )
        fig.update_layout(
            template=VisualConfig.THEME,
            height=max(550, 32 * matrix_dimension),
            width=max(550, 32 * matrix_dimension)
        )
        fig.show()


# -------------------------------------------------------------
# APPLICATION ENTRYPOINT / EXECUTION WORKFLOW
# -------------------------------------------------------------
if __name__ == "__main__":
    print("=" * 60)
    print("       DATA ARCHITECT PIPELINE INITIALIZATION       ")
    print("=" * 60)

    # Initialize Engine Object
    architect = AdvancedDataArchitect()

    # Load Demo Data (Titanic Repo Source)
    print("\n[INIT] Fetching remote Titanic repository sample data...")
    source_endpoint = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    architect.dataframe = pd.read_csv(source_endpoint)
    architect.source_dataframe = architect.dataframe.copy()
    architect._refresh_feature_types()

    # Process pipeline functions
    architect.generate_structural_audit()
    architect.smart_imputation_engine(strategy='auto')
    architect.purge_duplicate_records()
    architect.manage_outlier_variance(target_cols=['Age', 'Fare'], action='cap')

    # Execute Transformation logic blocks
    numerical_scaled_space = architect.scale_numerical_space(technique='robust', feature_subsets=['Age', 'Fare'])
    categorical_encoded_space = architect.encode_categorical_space(technique='onehot', feature_subsets=['Sex', 'Embarked'])
    integrated_master_frame = architect.merge_transformed_matrices(numerical_scaled_space, categorical_encoded_space)

    # Trigger Advanced Visual Plot Interfaces
    architect.visualization_univariate_grid(target_features=['Age', 'Fare'])
    architect.visualization_bivariate_relationship(var_x='Age', var_y='Fare')
    architect.visualization_bivariate_relationship(var_x='Sex', var_y='Survived')
    architect.visualization_categorical_distribution(feature='Pclass')
    architect.visualization_association_matrix()

    print(f"\n[COMPLETE] Script processed successfully. Matrix dimensions: {architect.dataframe.shape}")

       DATA ARCHITECT PIPELINE INITIALIZATION       

[INIT] Fetching remote Titanic repository sample data...

⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡
                  DATASET STRUCTURAL REPORT                 
⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡⚡
• Dynamic Shapes  : 891 records | 12 variables
• Numerical Pool  : 7 features
• Categorical Pool: 5 features

--- SAMPLE FRAMEWORK (TOP 5 DEEP INSPECTION) ---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4   


[COMPLETE] Script processed successfully. Matrix dimensions: (891, 12)
